[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/rare_variant_Fisher.ipynb)

# Fisher's exact tests

In this section, you'll be running a collection of Fisher's exact tests and plotting the results. The data you will be analysing represents a subset of genes in a bipolar exome sequencing analysis.

First we'll need to grab the data files that we need in order to carry out the practical.

The genetic and phenotypic data has been quality controlled. You can take a look at the quality control pipeline [here](https://astheeggeggs.github.io/BipEx/qc.html).

The cohort has been restricted to unrelated individuals with European genetic ancestry.

---

Quality control discussion

Briefly discuss the quality control carried out on the exome sequencing data in the linked pipeline above.

Take a look through and convince yourselves that these quality control steps are reasonable.

We've counted the number of individuals with an extremely rare (MAC < 5, and not present in the non-psych portion of gnomAD) pLoF or damaging missense variant in each gene, split by case status, and saved the results to a file named dataset_1.tsv.gz.

Take a look at these counts by using head or less on the file dataset_1.tsv.gz

First move to the folder:
cd ~/practicals/5.2.RareVariantAssociation_DuncanPalmer/final/Fisher

Then, take a look:
zless -S dataset_1.tsv.gz

Question 1.1. Using these data, what hypotheses might we want to test?

One series of tests that we could perform is to look for evidence of an association between presence of a pLoF in an individual and case status, at the gene-level.

Question 1.2. What class of variation could you use as a negative control?

- synonymous variation
- missense variation
- non-coding variation
- synonymous variation with MAC < 5, and not in gnomAD non-psych

Let's test for an association between presence of a pLoF in an individual and case status in a subset of these genes using Fisher's exact tests. In particular, we'll extract the collection of genes in gene_names_to_extract.tsv.gz.

Oh no, the file dataset_1.tsv.gz has the gene IDs saved as ensembl IDs. What idiot sent you this gene list anyway?

Question 1.3. Can you use the mapping file provided in the extdata folder (gene_mapping_to_extract.tsv.gz) to map the gene symbols onto ensembl IDs and then restrict to the genes in gene_names_to_extract.tsv.gz? Let's do this in R, so make sure you're in the 'console' tab in Rstudio.

Have a think and then click to the next slide for code to do this...

In [ ]:
# Load the R magic extension so we can write native R inside this notebook
%load_ext rpy2.ipython

In [ ]:
%%R
library(data.table)
library(dplyr)

setwd("~/practicals/5.2.RareVariantAssociation_DuncanPalmer/final/Fisher")

# Question 1.3
# Merge and restrict to the 100 genes listed in gene_names_to_extract.tsv.gz.

dt_to_extract <- fread("gene_names_to_extract.tsv.gz", header=FALSE)
dt_map <- fread("gene_mapping_to_extract.tsv.gz")
dt <- fread(file="dataset_1.tsv.gz", sep='\t')

dt_to_extract <- dt_to_extract %>% mutate(gene_name = V1)
setkey(dt_to_extract, "gene_name")
dt_map <- dt_map %>% dplyr::rename(gene_id = `Gene stable ID`, gene_name = `Gene name`)
setkey(dt_map, "gene_name")
dt_genes <- merge(dt_map, dt_to_extract) %>% dplyr::select(gene_id, gene_name)

setkey(dt_genes, "gene_id")
setkey(dt, "gene_id")

# Filter down to the set of genes present in the list passed
dt <- merge(dt_genes, dt)

Following the restriction, which are the first five gene names alphabetically?


In [ ]:
%%R
setorder(dt, "gene_name")
dt$gene_name[1:5]

Click to the next slide to see what you should have found!


The genes at the top of your list alphabetically should be...

ACTL6B
ADGRG3
AKAP11
ARL14EPL
ASAP1

Now, let's perform a series of Fisher's exact tests. First, we'll write a function to pass the data to.

You may assume that the number of cases and controls is fixed for each gene:

N cases = 13,933, N controls = 14,422.

Question 1.4. Given that information, can you create an R function that uses the two provided counts, runs a two-sided Fisher's exact test, and provides an odds ratio and P-value?

The function requires (or should deduce) the following information:
 	has pLoF	does not have pLoF
case	a	b
control	c	d

Note that you don't need to evaluate this yourself, you can make use of R functions. Take a look at fisher.test. If you're not familiar with R, you can check the help by running ?fisher.test


In [ ]:
%%R
run_fisher_test <- function(case_var, control_var, n_cases=13933, n_controls=14422) {
  # create a 2x2 table of the inputted numbers
  mat <- matrix(
    c(case_var, n_cases - case_var, control_var, n_controls-control_var), nrow=2)
  # perform a Fisher's exact test
  f <- fisher.test(mat, alternative='two.sided')
  # return the result
  return(f)
}

Now, let's test whether there is an association between presence of a pLoF in the gene and case status for each of the 100 genes in gene_names_to_extract.tsv.gz.

Generate a table containing the Gene name, P-value, and OR for each gene.


In [ ]:
%%R
dt[, c("p_value", "odds_ratio") := {
  test <- run_fisher_test(
    ptv_fisher_gnom_non_psych_case_count,
    ptv_fisher_gnom_non_psych_control_count
  )
  list(test$p.value, test$estimate)
}, by = seq_len(nrow(dt))]  # Apply per row
dt <- dt %>% dplyr::select(gene_id, gene_name, p_value, odds_ratio)


Now, sort the table by P-value from lowest to highest.


In [ ]:
%%R
setorder(dt, p_value)
head(dt, 5)

Question 1.5. What genes are in the top 5 rows of your table? Click on to see if you got it right!

The top of your table should look something like this...
```
           gene_id gene_name      p_value odds_ratio
            <char>    <char>        <num>      <num>
1: ENSG00000023516    AKAP11 1.150102e-05        Inf
2: ENSG00000083097     DOP1A 2.221591e-04 15.5442757
3: ENSG00000058804      NDC1 5.828607e-02        Inf
4: ENSG00000129595  EPB41L4A 7.231816e-02  2.8479521
5: ENSG00000011028      MRC2 1.802036e-01  0.2956758
```

Question 1.6. What would the Bonferroni significance threshold be, assuming that we would plan to test all of the genes, not just these 100? Hint: you should check the number of rows of the original table you read in


Your answer should be 0.05/19225

Question 1.7. So, are any associations significant?


If you said 'no', you're right!

Here's another file that contains the associated P-value for all of the genes: dataset_2.tsv.gz.

Question 1.8. Can you create a QQ plot of the P-values for these results?
Have a think of how you might construct code to do this, and click on for some code to copy.

In [ ]:
%%R
dt <- fread("dataset_2.tsv.gz")
plot(rev(-log10(seq(1, nrow(dt))/(nrow(dt)+1))),
     sort(-log10(dt$plof_p_value)), xlab='Expected -log(P)',
     ylab = 'Observed -log(P)')
abline(0,1,col='red')

QQ plot discussion

What does the QQ plot look like?
Does it seem reasonable?
How are you are assuming that P-values are sampled under the null?
Why do you think there are 'steps' in the QQ plot?

Click on!

It's below y=x, and looks 'steppy'
We're assuming that the null is uniform(0,1).
But, when counts are low, there are only so many P-values that the Fisher's exact test can take - it's discretized, so it's not actually uniform(0,1) under the null, hence the steps.


What happens if you filter to the set of genes for which at least 10 individuals have a variant in the gene?

Question 1.9. Create a QQ plot again, this time after filtering to this subset of genes. Does it look more 'calibrated'? Here's some code to that...

In [ ]:
%%R

dt <- dt %>% filter((plof_fisher_gnom_non_psych_control_count +
                       plof_fisher_gnom_non_psych_case_count) > 10)
plot(rev(-log10(seq(1, nrow(dt))/(nrow(dt)+1))),
     sort(-log10(dt$plof_p_value)), xlab='Expected -log(P)',
     ylab = 'Observed -log(P)')
abline(0,1,col='red')

Ah nice, much more like it!

Optional section

Another researcher took a look at the first dataset that you analysed and thought that it would be a great idea to also analyse damaging missense variation.

This seems like a good idea, but would require accounting for double the number of tests.

They suggest performing a Cauchy combination test, and controlling for the same number of tests. This seems like black magic. Can you really aggregate as many tests as you like, but maintain the same effective number of tests? Cheers Cauchy.

Using the following code to determine Cauchy combination P-values for each gene:


In [ ]:
%%R

CCT <- function(pvals, weights=NULL)
{
  if (sum(is.na(pvals)) > 0) {
    stop("Cannot have NAs in the p-values!")
  }

  if ((sum(pvals < 0) + sum(pvals > 1)) > 0){
    stop("All p-values must be between 0 and 1!")
  }

  is.zero <- (sum(pvals == 0) >= 1)
  is.one <- (sum(pvals == 1) >= 1)

  if (is.zero) { return(0) }

  if (is.one) {
    warning("There are p-values that are exactly 1!")
    return(min(1, min(pvals)*length(pvals)))
  }

  if (is.null(weights)){
    weights <- rep(1/length(pvals), length(pvals))
  } else if (length(weights) != length(pvals)) {
    stop("The length of weights should be the same as that of the p-values!")
  } else if (sum(weights < 0) > 0) {
    stop("All the weights must be positive!")
  } else {
    weights <- weights / sum(weights)
  }

  is.small <- pvals < 1e-16
  if (sum(is.small) == 0) {
    cct.stat <- sum(weights * tan((0.5 - pvals)*pi))
  } else {
    cct.stat <- sum((weights[is.small] / pvals[is.small]) / pi)
    cct.stat <- cct.stat +
      sum(weights[!is.small] * tan((0.5 - pvals[!is.small]) * pi))
  }

  if (cct.stat > 1e+15) {
    pval <- (1 / cct.stat) / pi
  } else {
    pval <- 1 - pcauchy(cct.stat)
  }
  return(pval)
}

dt <- fread("dataset_2.tsv.gz")
dt[, cauchy_p := CCT(c(plof_p_value, damaging_missense_p_value)), by = seq_len(nrow(dt))]

Questions 1.10. Are there any genes which become significant in the Cauchy combination test that were not significant in either the pLoF or damaging missense tests? Why?

There aren't. There are no associations that are exome-wide significant in the damaging missense class of variation, and the test statistic for the Cauchy combination test is such that it's impossible to get a lower P-value than the minimum of the P-values to be combined.

Question 1.11. What would happen if you Cauchy combined exactly the same P-values?

You'd just get back what you put in. See, for example:

In [ ]:
%%R
p <- runif(100)
dt <- data.table(p1 = p, p2 = p)
dt[, cauchy_p := CCT(c(p1, p2)), by = seq_len(nrow(dt))]

Question 1.12. Under what scenarios might it be a good idea to use a Cauchy combination test?

If you've got a bunch of tests that are correlated, but not the same. For example, when combining across different allele frequency bins.